In [1]:
import os

# ── Kaggle TPU v5e PJRT Workaround ───────────────────────────────────────────
# Remove Kaggle's conflicting environment variables before importing torch_xla
os.environ.pop('TPU_PROCESS_ADDRESSES', None)
os.environ.pop('CLOUD_TPU_TASK_ID', None)

import json, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import timm

SCALES_TO_RUN = ["10%", "25%", "50%", "100%"]   # edit per Kaggle account
EPOCHS        = 10
LR            = 5e-4
WEIGHT_DECAY  = 0.05

# Ready for downstream distillation notebooks (ResNet-50 2048-d pool -> DeiT 192-d)
PROJ_DIM_IN   = 192
PROJ_DIM_OUT  = 2048

In [2]:

DEVICE_TYPE = "TPU"

class DeviceManager:
    """Unified stub so the main training logic is identical for GPU and TPU."""
    def __init__(self):
        self.type = DEVICE_TYPE
        if self.type == "TPU":
            import torch_xla
            import torch_xla.core.xla_model as xm
            import torch_xla.runtime as xr
            import torch_xla.distributed.parallel_loader as pl
            self.xm = xm
            self.xr = xr
            self.pl = pl
            self.device = torch_xla.device()
            self.world_size = xr.world_size()
            self.rank = xr.global_ordinal()
        else:
            self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            self.world_size = 1
            self.rank = 0

    def step(self, opt):
        if self.type == "TPU":
            self.xm.optimizer_step(opt)
            self.xm.mark_step()
        else:
            opt.step()

    def wrap_loader(self, loader):
        if self.type == "TPU":
            return self.pl.MpDeviceLoader(loader, self.device)
        return loader

    def is_master(self):
        return self.rank == 0

    def master_print(self, *args, **kwargs):
        if self.is_master():
            print(*args, **kwargs)

In [3]:
TRAIN_DIR    = "/kaggle/input/datasets/melikechan/cifar100/cifar100/train"
TEST_DIR     = "/kaggle/input/datasets/melikechan/cifar100/cifar100/test"

# Updated to reflect ResNet-50 Teacher with 2048-d avgpool
TEACHER_CKPT = "/kaggle/input/datasets/totallyapoorv/resnet-teachermodel-50/teacher_resnet50.pth"

WORK_DIR = "./outputs"
# Removed DeviceManager instantiation to prevent early XLA runtime initialization.
# exist_ok=True safely handles parallel directory creation.
os.makedirs(f"{WORK_DIR}/checkpoints", exist_ok=True)
os.makedirs(f"{WORK_DIR}/results",     exist_ok=True)

In [4]:
SEED       = 67
BATCH_SIZE = 64
scale_map  = {"10%": 0.10, "25%": 0.25, "50%": 0.50, "100%": 1.0}
key_of     = lambda s: s.replace("%", "pct")

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

def build_loaders(scale, ctx):
    tfm_tr = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.5071,0.4867,0.4408),(0.2675,0.2565,0.2761)),
    ])
    tfm_te = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize((0.5071,0.4867,0.4408),(0.2675,0.2565,0.2761)),
    ])
    
    full = torchvision.datasets.ImageFolder(TRAIN_DIR, transform=tfm_tr)
    test = torchvision.datasets.ImageFolder(TEST_DIR,  transform=tfm_te)
    
    idx = list(range(len(full)))
    random.Random(SEED).shuffle(idx)
    sub = torch.utils.data.Subset(full, idx[:int(len(full)*scale_map[scale])])
    
    # Required for TPU distributed parallel scaling
    sampler = torch.utils.data.distributed.DistributedSampler(
        sub, num_replicas=ctx.world_size, rank=ctx.rank, shuffle=True
    ) if ctx.world_size > 1 else None

    tr = torch.utils.data.DataLoader(
        sub, BATCH_SIZE, shuffle=(sampler is None), 
        sampler=sampler, num_workers=2, pin_memory=True
    )
    te = torch.utils.data.DataLoader(test, BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    return tr, te, len(sub)

In [5]:
def evaluate(model, loader, ctx):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in ctx.wrap_loader(loader):
            x, y = x.to(ctx.device), y.to(ctx.device)
            out = model(x)
            if isinstance(out, tuple): out = (out[0] + out[1]) / 2
            correct += out.argmax(1).eq(y).sum().item()
            total   += y.size(0)
    
    if ctx.type == "TPU":
        correct = ctx.xm.mesh_reduce("test_correct", correct, sum)
        total = ctx.xm.mesh_reduce("test_total", total, sum)
        
    return 100. * correct / total

def save_ckpt(path, epoch, model, proj, opt, best_acc, history, extras=None):
    d = dict(epoch=epoch, model=model.state_dict(), opt=opt.state_dict(),
             best_acc=best_acc, history=history)
    if proj is not None: d["proj"] = proj.state_dict()
    if extras:           d.update(extras)
    torch.save(d, path)

def load_ckpt(path, model, proj, opt, ctx):
    d  = torch.load(path, map_location=ctx.device)
    model.load_state_dict(d["model"])
    if proj is not None and "proj" in d: proj.load_state_dict(d["proj"])
    opt.load_state_dict(d["opt"])
    return d["epoch"]+1, d["best_acc"], d.get("history", [])

In [6]:
def build_student(ctx):
    # set_distilled_training stays False (default) — no dual-head split needed.
    m = timm.create_model("deit_tiny_distilled_patch16_224", 
                          pretrained=False, num_classes=100)
    m = m.to(ctx.device)
    if ctx.is_master():
        total_params = sum(p.numel() for p in m.parameters())
        print(f"  Student params: {total_params/1e6:.2f}M")
    return m

In [7]:
def train_one_scale(index, scale):
    ctx = DeviceManager()
    key  = key_of(scale)
    ckpt = f"{WORK_DIR}/checkpoints/nb01_{key}.pth"
    rp   = f"{WORK_DIR}/results/nb01_{key}.json"

    if os.path.exists(rp) and json.load(open(rp)).get("completed"):
        r = json.load(open(rp))
        ctx.master_print(f"[{scale}] already done — best {r['best_acc']:.2f}%")
        return r

    tr_loader, te_loader, n_train = build_loaders(scale, ctx)
    model = build_student(ctx)
    opt   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    start_epoch, best_acc, history = 0, 0.0, []
    if os.path.exists(ckpt):
        start_epoch, best_acc, history = load_ckpt(ckpt, model, None, opt, ctx)
        ctx.master_print(f"[{scale}] resumed at epoch {start_epoch+1}")

    ctx.master_print(f"[{scale}] training on {n_train} images")
    for epoch in range(start_epoch, EPOCHS):
        if hasattr(tr_loader, 'sampler') and hasattr(tr_loader.sampler, 'set_epoch'):
            tr_loader.sampler.set_epoch(epoch)
            
        model.train()
        t0 = time.time()
        total_loss = 0.0

        for x, y in ctx.wrap_loader(tr_loader):
            x, y = x.to(ctx.device), y.to(ctx.device)
            opt.zero_grad()
            out  = model(x)
            loss = F.cross_entropy(out, y)
            loss.backward()
            ctx.step(opt)
            total_loss += loss.item()

        epoch_time = time.time() - t0
        
        # Reduce logging across mesh
        if ctx.type == "TPU":
            avg_loss = ctx.xm.mesh_reduce("loss_reduce", total_loss, sum) / (len(tr_loader) * ctx.world_size)
        else:
            avg_loss = total_loss / len(tr_loader)
            
        val_acc  = evaluate(model, te_loader, ctx)
        best_acc = max(best_acc, val_acc)

        if ctx.is_master():
            history.append({"epoch": epoch, "ce_loss": avg_loss,
                            "val_acc": val_acc, "epoch_time": epoch_time})
            ctx.master_print(f"[{scale}] E{epoch+1}/{EPOCHS} | CE={avg_loss:.4f} | "
                             f"val={val_acc:.2f}% | {epoch_time:.0f}s")
            save_ckpt(ckpt, epoch, model, None, opt, best_acc, history)

    if ctx.is_master():
        result = {"scale": scale, "n_train": n_train, "final_acc": history[-1]["val_acc"] if history else best_acc,
                  "best_acc": best_acc, "history": history, "completed": True}
        json.dump(result, open(rp,"w"), indent=2)
        ctx.master_print(f"[{scale}] DONE — final {result['final_acc']:.2f}% | best {best_acc:.2f}%")
        return result
    return None

In [ ]:
def _mp_fn(index, scales):
    for scale in scales:
        train_one_scale(index, scale)

if __name__ == "__main__":
    if DEVICE_TYPE == "TPU":
        import torch_xla.distributed.xla_multiprocessing as xmp
        xmp.spawn(_mp_fn, args=(SCALES_TO_RUN,), start_method='fork')
    else:
        _mp_fn(0, SCALES_TO_RUN)
    print("All scales complete.")

/usr/local/lib/python3.12/site-packages/torch_xla/__init__.py:258: UserWarning: `tensorflow` can conflict with `torch-xla`. Prefer `tensorflow-cpu` when using PyTorch/XLA. To silence this warning, `pip uninstall -y tensorflow && pip install tensorflow-cpu`. If you are in a notebook environment such as Colab or Kaggle, restart your notebook runtime afterwards.
  warnings.warn(
/usr/local/lib/python3.12/site-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/site-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  Student params: 5.56M
[10%] training on 5000 images


/usr/local/lib/python3.12/site-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/site-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/site-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/site-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/site-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument i

[10%] E1/10 | CE=4.4747 | val=4.28% | 76s
[10%] E2/10 | CE=4.3188 | val=5.10% | 3s
[10%] E3/10 | CE=4.2315 | val=5.54% | 3s
[10%] E4/10 | CE=4.1619 | val=5.58% | 3s
[10%] E5/10 | CE=4.1073 | val=6.01% | 3s
[10%] E6/10 | CE=4.0211 | val=7.82% | 3s
[10%] E7/10 | CE=3.9304 | val=8.99% | 3s
[10%] E8/10 | CE=3.8516 | val=9.37% | 3s
[10%] E9/10 | CE=3.7885 | val=10.05% | 3s
[10%] E10/10 | CE=3.7326 | val=11.49% | 3s
[10%] DONE — final 11.49% | best 11.49%


In [ ]:
print(f"\n{'Scale':<8} {'Final Acc':>10} {'Best Acc':>10} {'Train (s)':>12}")
for scale in SCALES_TO_RUN:
    rp = f"{WORK_DIR}/results/nb01_{key_of(scale)}.json"
    if os.path.exists(rp):
        r = json.load(open(rp))
        wall = sum(h.get("epoch_time",0) for h in r["history"])
        print(f"{scale:<8} {r['final_acc']:>9.2f}% {r['best_acc']:>9.2f}% {wall:>11.0f}s")
    else:
        print(f"{scale:<8} not yet complete")